# Part 6: Streaming API (Problems 043–050)

This part builds the HTTP layer that exposes your inference engine to the world:

- **Problem 043**: Define the `GenerateRequest` Pydantic schema with validators
- **Problem 044**: Define the `GenerateResponse` schema
- **Problem 045**: Build the inference engine singleton
- **Problem 046**: Implement the streaming token generator (async)
- **Problem 047**: Build the SSE streaming endpoint (`POST /generate/stream`)
- **Problem 048**: Build the non-streaming endpoint (`POST /generate`)
- **Problem 049**: Add request ID tracking
- **Problem 050**: Test streaming with curl

### What is Server-Sent Events (SSE)?

SSE is a simple HTTP streaming protocol where the server sends data events to the client as they become available:

```
HTTP/1.1 200 OK
Content-Type: text/event-stream

data: The

data: quick

data: brown

data: fox

data: [DONE]

```

Each `data:` line delivers one token. The client reads them as they arrive, enabling word-by-word streaming like you see in ChatGPT.

## Cell 1: Define and validate a GenerateRequest

In [ ]:
import sys
from pathlib import Path

# Make sure the project root is on sys.path so solutions/ is importable
project_root = Path('__file__').parent.parent if '__file__' in dir() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# Also try the current directory's parent
for p in [Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / 'solutions').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break


In [ ]:
import importlib

try:
    _m = importlib.import_module("solutions.043_define_generate_request_schema")
    GenerateRequest = _m.GenerateRequest
except Exception:
    print("Solve problem 043 first:")
    print("  cp problems/043_define_generate_request_schema.py solutions/043_define_generate_request_schema.py")
    GenerateRequest = None

if GenerateRequest is not None:
    # Valid request
    req = GenerateRequest(
        prompt="Explain paged attention in simple terms",
        max_tokens=150,
        temperature=0.7,
        top_k=40,
        top_p=0.9,
    )
    print("Valid request:")
    print(f"  prompt      : '{req.prompt[:50]}...'")
    print(f"  max_tokens  : {req.max_tokens}")
    print(f"  temperature : {req.temperature}")
    print(f"  top_k       : {req.top_k}")
    print(f"  top_p       : {req.top_p}")
    print()

    # Test validation
    from pydantic import ValidationError

    invalid_cases = [
        ({"prompt": "hi", "temperature": 0.0}, "temperature=0.0 should fail"),
        ({"prompt": "hi", "max_tokens": -5}, "max_tokens=-5 should fail"),
        ({"prompt": "hi", "max_tokens": 9999}, "max_tokens=9999 should fail (>2048)"),
        ({"prompt": "hi", "top_k": -1}, "top_k=-1 should fail"),
        ({"prompt": "hi", "top_p": 1.5}, "top_p=1.5 should fail"),
    ]

    print("Validation tests:")
    for kwargs, description in invalid_cases:
        try:
            GenerateRequest(**kwargs)
            print(f"  FAIL: {description} (no error raised)")
        except (ValidationError, ValueError) as e:
            print(f"  PASS: {description}")

## Cell 2: Show the SSE format — what a streaming response looks like

In [ ]:
import importlib
import asyncio

try:
    _m = importlib.import_module("solutions.046_implement_streaming_generator")
    streaming_generator = _m.streaming_generator
except Exception:
    print("Solve problem 046 first:")
    print("  cp problems/046_implement_streaming_generator.py solutions/046_implement_streaming_generator.py")
    streaming_generator = None

print("SSE (Server-Sent Events) format:")
print()
print("  Each event is formatted as:")
print("    data: <content>\\n\\n")
print()
print("  Example response for prompt='The quick brown':")
print("  ---")
example_tokens = ["fox", " jumped", " over", " the", " lazy", " dog", ".", "[DONE]"]
for token in example_tokens:
    print(f"  data: {token}")
    print()
print("  ---")

if streaming_generator is not None:
    print()
    print("Testing your streaming_generator:")

    async def test_generator():
        request = {
            "prompt": "The quick brown fox",
            "max_tokens": 5,
            "temperature": 1.0,
        }
        events = []
        async for event in streaming_generator(request):
            events.append(event)
        return events

    events = asyncio.run(test_generator())
    print(f"  Generated {len(events)} SSE events:")
    for ev in events:
        print(f"    {repr(ev)}")

## Cell 3: Start the server with uvicorn and send a test request

In [ ]:
import importlib

try:
    _m = importlib.import_module("solutions.048_build_non_streaming_endpoint")
    create_app = _m.create_app
except Exception:
    print("Solve problem 048 first:")
    print("  cp problems/048_build_non_streaming_endpoint.py solutions/048_build_non_streaming_endpoint.py")
    create_app = None

try:
    _m = importlib.import_module("solutions.047_build_sse_streaming_endpoint")
    create_streaming_app = _m.create_streaming_app
except Exception:
    print("Solve problem 047 first:")
    print("  cp problems/047_build_sse_streaming_endpoint.py solutions/047_build_sse_streaming_endpoint.py")
    create_streaming_app = None

if create_app is not None:
    # Use FastAPI's TestClient for in-process testing (no actual server needed)
    from fastapi.testclient import TestClient

    app = create_app()
    client = TestClient(app)

    print("Testing POST /generate (non-streaming):")
    resp = client.post(
        "/generate",
        json={"prompt": "The future of AI is", "max_tokens": 10}
    )
    print(f"  Status: {resp.status_code}")
    if resp.status_code == 200:
        data = resp.json()
        print(f"  generated_text    : '{data.get('generated_text', '?')}'")
        print(f"  tokens_generated  : {data.get('tokens_generated', '?')}")
        print(f"  time_to_first_token: {data.get('time_to_first_token', '?'):.6f}s")
        print(f"  total_time        : {data.get('total_time', '?'):.6f}s")
        print(f"  request_id        : {data.get('request_id', '?')}")

if create_streaming_app is not None:
    from fastapi.testclient import TestClient

    streaming_app = create_streaming_app()
    streaming_client = TestClient(streaming_app)

    print()
    print("Testing POST /generate/stream (SSE):")
    resp = streaming_client.post(
        "/generate/stream",
        json={"prompt": "Once upon a time", "max_tokens": 8}
    )
    print(f"  Status: {resp.status_code}")
    print(f"  Content-Type: {resp.headers.get('content-type', '?')}")
    lines = [l for l in resp.text.splitlines() if l.startswith("data:")]
    print(f"  Events received: {len(lines)}")
    for line in lines[:5]:
        print(f"    {line}")
    if len(lines) > 5:
        print(f"    ... ({len(lines)-5} more)")

## Cell 4: Test with curl

In [ ]:
# These are the curl commands you can run once the server is started with:
#   python3 main.py

print("=" * 60)
print("Server startup:")
print("  python3 main.py")
print()
print("=" * 60)
print("Non-streaming (returns full response):")
print()
print("curl -s -X POST http://localhost:8000/generate \\")
print("  -H 'Content-Type: application/json' \\")
print("  -d '{")
print('    "prompt": "The quick brown fox",')  
print('    "max_tokens": 20,')  
print('    "temperature": 0.8,')  
print('    "top_k": 50,')  
print('    "top_p": 0.95')  
print("  }' | python3 -m json.tool")
print()
print("=" * 60)
print("Streaming (tokens arrive one by one):")
print()
print("curl -s -N -X POST http://localhost:8000/generate/stream \\")
print("  -H 'Content-Type: application/json' \\")
print("  -d '{")
print('    "prompt": "Once upon a time in a land",')  
print('    "max_tokens": 30,')  
print('    "temperature": 0.9')  
print("  }'")
print()
print("  -N flag disables buffering so you see tokens as they arrive!")
print()
print("=" * 60)
print("Health check:")
print()
print("curl http://localhost:8000/health")
print()
print("=" * 60)
print("Interactive API docs:")
print()
print("  http://localhost:8000/docs")

## Cell 5: Request tracking stats

In [ ]:
import importlib
import uuid
import time

try:
    _m = importlib.import_module("solutions.049_add_request_id_tracking")
    track_request = getattr(_m, "track_request", None)
    get_request_stats = getattr(_m, "get_request_stats", None)
except Exception:
    print("Solve problem 049 first:")
    print("  cp problems/049_add_request_id_tracking.py solutions/049_add_request_id_tracking.py")
    track_request = get_request_stats = None

if track_request is not None:
    # Simulate tracking 5 requests
    request_ids = []
    for i in range(5):
        req_id = str(uuid.uuid4())
        start = time.perf_counter()
        # Simulate some processing time
        time.sleep(0.01 * (i + 1))
        elapsed = time.perf_counter() - start
        track_request(req_id, tokens_generated=10 + i*5, latency=elapsed)
        request_ids.append(req_id)

    if get_request_stats is not None:
        stats = get_request_stats()
        print("Request tracking stats:")
        print(f"  Total requests     : {stats.get('total_requests', '?')}")
        print(f"  Total tokens       : {stats.get('total_tokens', '?')}")
        print(f"  Avg latency (ms)   : {stats.get('avg_latency_ms', '?'):.2f}")
        print(f"  Tokens/sec         : {stats.get('tokens_per_second', '?'):.1f}")
else:
    print("Request tracking would look like this:")
    print()
    print("  Total requests   : 1,247")
    print("  Total tokens     : 48,920")
    print("  Avg latency (ms) : 142.3")
    print("  Tokens/sec       : 84.7")
    print()
    print("Implement problem 049 to track your server's real performance.")